In [1]:
import os
from os import path
from datetime import datetime
import pandas as pd
import numpy as np
import pytz
from utils.parsing import load_properties, hour_rounder

In [2]:
dataset_id = 202
data_folder = path.join('./data', str(dataset_id))

In [3]:
all_users = pd.read_csv(path.join(data_folder, "participants.csv"), low_memory=False) 
all_events = pd.read_csv(path.join(data_folder, "histories.csv"), low_memory=False)
all_survey_answers = pd.read_csv(path.join(data_folder, "survey-answers.csv"), low_memory=False) 

tprops_filename = path.join(data_folder, "time.properties")
tprops = load_properties(tprops_filename)
sim_tz = tprops['sim_tz']
time0 = tprops['time0']
time1 = tprops['time1']
timezone = pytz.timezone(sim_tz)

In [4]:
# Getting list of users who quitted so they are removed from the data

outcomes = all_events[all_events["type"] == "outcome"]
qlist = {}
elist = {}
dlist = {}
for uid, t, o in zip(outcomes.user_id.values, outcomes.time.values, outcomes.out.values):
    if o == 'DEAD':
        dlist[uid] = t
    if o == 'ESCAPED':
        elist[uid] = t
    if o == 'QUIT':
        qlist[uid] = t

uids_to_remove = []
for uid in list(qlist.keys()):
    if uid in elist:        
        if elist[uid] < qlist[uid]:
            # If we are here, it means that the user first escaped the epigame 
            # and then quit from the escape screen
            print(uid, 'escaped before quitting, so must have quit from the escaped screen')
        else:
            # If we are here, it means that the user first quit the epigame 
            # and then rejoined before escaping            
            print(uid, 'escaped after quitting, so must have rejoined')
    elif uid in dlist:
        if dlist[uid] < qlist[uid]:
            # If we are here, it means that the user first died in the epigame 
            # and then quit from the dead screen
            print(uid, 'died before quitting, so must have quit from the dead screen')
        else:
            # If we are here, it means that the user first quit the epigame 
            # and then rejoined before dying            
            print(uid, 'died after quitting, so must have rejoined')
    else:
        # Only the users here are consdiered to have truly left the study midway through
        uids_to_remove += [uid]

9876 escaped after quitting, so must have rejoined
9958 escaped after quitting, so must have rejoined
10110 escaped before quitting, so must have quit from the escaped screen
9944 escaped before quitting, so must have quit from the escaped screen


In [5]:
for uid in uids_to_remove:
    rid = all_users[all_users['id'] == uid]['random_id'].values[0]
    print(uid, rid, hour_rounder(datetime.fromtimestamp(qlist[uid], tz=timezone)))

9818 1343 2025-11-16 13:00:00+03:00
10055 9815 2025-11-17 12:00:00+03:00
10180 5000 2025-11-18 09:00:00+03:00
9870 4362 2025-11-18 14:00:00+03:00
10282 1650 2025-11-22 00:00:00+03:00
10302 4236 2025-11-24 10:00:00+03:00


In [6]:
# Looking into participants with no group assignment

uids_no_group = list(all_users[all_users['group'].isna()]['id'].values)
print('Participants with no group assignment =', len(uids_no_group))
print()

uids_inferred_in_group1 = []
uids_inferred_in_group2 = []

for uid in uids_no_group:
    user_events = all_events[all_events['user_id'] == uid]
    qtevents = user_events[(user_events['inf'] == 'quarantine')]
    nqtevents = user_events[(user_events['inf'] == 'noQuarantine')]

    # print('User', uid, 'has no group assignment')
    # print('  Total number of events =', len(user_events))
    # print('  Total number of quarantine events =', len(qtevents))
    # print('  Quarantine points =', qtevents.out.values)
    # print('  Total number of non-quarantine events =', len(nqtevents))
    # print('  Non-quarantine points =', nqtevents.out.values)

    arr = nqtevents.out.values
    if arr.size > 0 and np.all(arr == '10'):
        print(uid, arr)
        uids_inferred_in_group1 += [int(uid)]

    if arr.size > 0 and np.all(arr == '17'):
        print(uid, arr)
        uids_inferred_in_group2 += [int(uid)]

print('Participants inferred to be in Group 1 =', uids_inferred_in_group1)
print('Participants inferred to be in Group 2 =', uids_inferred_in_group2)

Participants with no group assignment = 31

9806 ['17' '17']
9767 ['10' '10' '10' '10' '10' '10' '10' '10' '10']
9984 ['17' '17' '17' '17']
9929 ['10']
10011 ['17' '17' '17' '17' '17' '17' '17' '17']
10081 ['10' '10' '10' '10' '10']
10075 ['10']
10031 ['10']
10283 ['10' '10']
Participants inferred to be in Group 1 = [9767, 9929, 10081, 10075, 10031, 10283]
Participants inferred to be in Group 2 = [9806, 9984, 10011]


In [18]:
# Assigning the inferred groups back to the participants table

all_users.loc[all_users['id'].isin(uids_inferred_in_group1), 'group'] = 'group1'
all_users.loc[all_users['id'].isin(uids_inferred_in_group2), 'group'] = 'group2'


uids_no_group = [uid for uid in uids_no_group if uid not in uids_inferred_in_group1]
uids_no_group = [uid for uid in uids_no_group if uid not in uids_inferred_in_group2]

print('Participants still with no group assignment =', all_users['group'].isna().sum())
print(len(uids_no_group))

uids_to_remove += uids_no_group


Participants still with no group assignment = 22
22


In [20]:
# Determining IDs of contacts or infections caused by the users who quit
eids_to_remove = []

contacts = all_events[all_events["type"] == "contact"]
cont_to_remove = 0
for eid, pid in zip(contacts.id.values, contacts.peer_id.values):
    if pid in uids_to_remove:
        cont_to_remove += 1
        eids_to_remove += [eid]

infections = all_events[all_events["type"] == "infection"]
inf_to_remove = 0
for eid, inf in zip(infections.id.values, infections.inf.values):
    if 'PEER' in inf:
        id0 = int(inf[inf.index("[") + 1:inf.index(":")]) 
        if id0 in uids_to_remove:
            inf_to_remove += 1
            eids_to_remove += [eid]

print('Contacts to remove =', cont_to_remove)
print('Infections to remove =', inf_to_remove)

Contacts to remove = 4322
Infections to remove = 1


In [21]:
group1_ids = list(all_users[all_users['group'] == 'group1']['id'].values)
group2_ids = list(all_users[all_users['group'] == 'group2']['id'].values)

In [22]:
print('Number of participants assigned G1 =', len(group1_ids))
print('Number of participants who quit in G1 =', len(group2_ids))

Number of participants assigned G1 = 246
Number of participants who quit in G1 = 299


In [23]:
quit_g1 = 0
quit_g2 = 0
for uid in uids_to_remove:
    if uid in group1_ids:
        quit_g1 += 1
    if uid in group2_ids:
        quit_g2 += 1

print('Number of participants who quit in G1 =', quit_g1)
print('Number of participants who quit in G2 =', quit_g2)

Number of participants who quit in G1 = 2
Number of participants who quit in G2 = 4


In [24]:
g1_s1_count = 0
g1_s2_count = 0
g1_s3_count = 0

g2_s1_count = 0
g2_s2_count = 0
g2_s3_count = 0

for uid in group1_ids:
    # if uid in uids_to_remove: continue
    answers = all_survey_answers[all_survey_answers['user_id'] == uid]
    if 3 in answers.survey_id.values:
        g1_s1_count += 1
    if 4 in answers.survey_id.values:
        g1_s2_count += 1
    if 5 in answers.survey_id.values:
        g1_s3_count += 1
        
for uid in group2_ids:
    # if uid in uids_to_remove: continue
    answers = all_survey_answers[all_survey_answers['user_id'] == uid]
    if 3 in answers.survey_id.values:
        g2_s1_count += 1
    if 4 in answers.survey_id.values:
        g2_s2_count += 1
    if 5 in answers.survey_id.values:
        g2_s3_count += 1

print('In Group 1:')
print('  Number of participants who took S1 =', g1_s1_count)
print('  Number of participants who took S2 =', g1_s2_count)
print('  Number of participants who took S3 =', g1_s3_count)
print('In Group 2:')
print('  Number of participants who took S1 =', g2_s1_count)
print('  Number of participants who took S2 =', g2_s2_count)
print('  Number of participants who took S3 =', g2_s3_count)

In Group 1:
  Number of participants who took S1 = 183
  Number of participants who took S2 = 129
  Number of participants who took S3 = 22
In Group 2:
  Number of participants who took S1 = 209
  Number of participants who took S2 = 147
  Number of participants who took S3 = 26


## Saving cleaned data

In [25]:
# Removing entries in users, events, and survey answers directly related to these users
users = all_users[~all_users['id'].isin(uids_to_remove)].copy()
events = all_events[~((all_events['user_id'].isin(uids_to_remove)) | (all_events['id'].isin(eids_to_remove)))].copy()
survey_answers = all_survey_answers[~all_survey_answers['user_id'].isin(uids_to_remove)].copy()

In [26]:
print("Number of users removed:", len(all_users) - len(users))
print("Number of events removed:", len(all_events) - len(events))
print("Number of answers removed:", len(all_survey_answers) - len(survey_answers))

Number of users removed: 28
Number of events removed: 9781
Number of answers removed: 46


In [27]:
# Fixing types
col_to_convert = ['max_strength', 'score', 'quiz_id', 'contact_length', 'peer_id']
for col in col_to_convert:    
    events[col] = events[col].astype('Int64')

events['lat'] = None
events['lng'] = None

In [28]:
users.to_csv(path.join(data_folder, 'participants_cleaned.csv'), index=False)
events.to_csv(path.join(data_folder, 'histories_cleaned.csv'), index=False)
survey_answers.to_csv(path.join(data_folder, 'survey-answers_cleaned.csv'), index=False)